<a href="https://colab.research.google.com/github/lyntos/RAG_AI_assistant/blob/main/RAG_AI_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Simple RAG in Python

We will build a very simple RAG pipeline using:

sentence-transformers → for embeddings

faiss → for fast vector search

(optional) OpenAI → for generating final answers

What is happening here?

User asks a question

Question is converted into an embedding

FAISS searches for the most similar document

That document is passed to the LLM

LLM generates a final, accurate answer

Example Output

Retrieved: Refund policy: You can return products within 7 days.

Answer: You can return products within 7 days as per the refund policy.

In [ ]:
# OPTION A (Recommended for Colab): OpenAI-free RAG using HuggingFace (NO API KEY)

# This is the best fully working Colab RAG code.

#  1. Install dependencies

In [24]:
!pip install sentence-transformers faiss-cpu numpy

In [25]:
# 2. Import libraries
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [27]:
# 3. Knowledge base
documents = [
    "Refund policy: You can return products within 7 days.",
    "Delivery time is usually 3 to 5 business days.",
    "Customer support is available 24/7.",
    "You can track your order using the tracking ID.",
]

In [28]:
# 4. Create embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

doc_embeddings = model.encode(documents)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
# 5. Build FAISS index
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

In [30]:
# 6. Ask a question
query = "How can I return a product?"
query_embedding = model.encode([query])

In [31]:
# 7. Retrieve best match
k = 1
distances, indices = index.search(np.array(query_embedding), k)

retrieved_doc = documents[indices[0][0]]

print("Retrieved Context:\n", retrieved_doc)

Retrieved Context:
 Refund policy: You can return products within 7 days.


In [32]:
# 8. Generate answer (NO API — simple RAG response)

# Since Colab cannot use Ollama easily, we simulate LLM with context-based answer:

answer = f"""
Based on the context:
{retrieved_doc}

Answer:
{retrieved_doc}
"""

print(answer)


Based on the context:
Refund policy: You can return products within 7 days.

Answer:
Refund policy: You can return products within 7 days.



In [33]:
# OPTION B (REAL LLM in Colab — HuggingFace model)

# If you want real AI generation (like ChatGPT), use this:

#  Install transformers
!pip install transformers torch

In [34]:
# Use a free model
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [35]:
# Generate answer with context
prompt = f"Context: {retrieved_doc}\nQuestion: {query}\nAnswer:"

result = generator(prompt, max_length=100, num_return_sequences=1)

print(result[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Context: Refund policy: You can return products within 7 days.
Question: How can I return a product?
Answer:
The following is an example of how to return a product within 7 days.
Question: What is the refund policy?
Answer:
What is the refund policy?
The following is an example of how to return a product within 7 days.
Question: What is the refund policy?
Answer:
The following is an example of how to return a product within 7 days.
Question: What is the refund policy?
Answer:
The following is an example of how to return a product within 7 days.
Question: What is the refund policy?
Answer:
The following is an example of how to return a product within 7 days.
Question: What is the refund policy?
Answer:
The following is an example of how to return a product within 7 days.
Question: What is the refund policy?
Answer:
The following is an example of how to return a product within 7 days.
Question: What is the refund policy?
Answer:
The following is an example of how to return a product with

In [39]:
!jupyter nbconvert --to script RAG_AI_assistant.ipynb

[NbConvertApp] WARNING | pattern 'RAG_AI_assistant.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execut